# MedAssist AI — RAG-Based Medical Customer Support Assistant

In [5]:
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf langgraph transformers accelerate

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import pipeline
from typing import TypedDict
from langgraph.graph import StateGraph, END

## Upload hospital_policy.pdf in Colab before running next cells

In [7]:
loader = PyPDFLoader("hospital_policy.pdf")
documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 18


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 478


In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_6768/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("ChromaDB Created Successfully")

ChromaDB Created Successfully


In [11]:
generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
class GraphState(TypedDict):
    query: str
    context: str
    answer: str
    intent: str
    escalate: bool

In [13]:
def detect_intent(state):

    query = state["query"].lower()

    emergency_keywords = [
        "heart attack",
        "emergency",
        "critical",
        "bleeding",
        "unconscious"
    ]

    insurance_keywords = [
        "insurance",
        "claim",
        "policy",
        "coverage"
    ]

    if any(word in query for word in emergency_keywords):
        state["intent"] = "emergency"
        state["escalate"] = True

    elif any(word in query for word in insurance_keywords):
        state["intent"] = "insurance"
        state["escalate"] = False

    else:
        state["intent"] = "general"
        state["escalate"] = False

    return state

In [14]:
def retrieve_context(state):

    docs = retriever.invoke(state["query"])

    context = "\n".join([doc.page_content for doc in docs])

    state["context"] = context

    return state

In [15]:
def generate_answer(state):

    prompt = f"""
    You are a medical customer support assistant.

    Answer the question using the provided context.

    Context:
    {state['context']}

    Question:
    {state['query']}
    """

    result = generator(
        prompt,
        max_length=200,
        num_return_sequences=1
    )

    state["answer"] = result[0]["generated_text"]

    return state

In [16]:
def human_escalation(state):

    if state["escalate"]:
        state["answer"] = (
            "This query requires immediate human medical assistance. "
            "Your request has been escalated to a hospital support agent."
        )

    return state

In [17]:
workflow = StateGraph(GraphState)

workflow.add_node("intent_detection", detect_intent)
workflow.add_node("retrieval", retrieve_context)
workflow.add_node("generation", generate_answer)
workflow.add_node("hitl", human_escalation)

workflow.set_entry_point("intent_detection")

workflow.add_edge("intent_detection", "retrieval")
workflow.add_edge("retrieval", "generation")
workflow.add_edge("generation", "hitl")
workflow.add_edge("hitl", END)

app = workflow.compile()

In [18]:
query = "Can I claim insurance for MRI scan?"

response = app.invoke({
    "query": query,
    "context": "",
    "answer": "",
    "intent": "",
    "escalate": False
})

print("Intent:", response["intent"])
print("\nAnswer:\n")
print(response["answer"])

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Intent: insurance

Answer:


    You are a medical customer support assistant.

    Answer the question using the provided context.

    Context:
    24.  Second Medical Opinion:  The Insured Person can obtain a Second Medical Opinion from a Doctor in the Company’s 
network of Medical Practitioners. All the medical records provided by the Insured Person will be submitted to the Doctor 
chosen by him/her online and the medical opinion will be made available directly to the Insured by the Doctor. To utilize this 
beneﬁt, all medical records should be forwarded to the mail-id e_medicalopinion@starhealth.in or through Post/Courier.
congenital disorders) or accidental injuries are payable from Day 1 of its birth till the expiry date of the policy.
Note: This cover is available only If Delivery Expenses Claim is paid under this policy or if Mother is covered under this
policy for a continuous period of 12 months without break 
Sum Insured in Lakhs (Rs.) Limit Per Policy Period (Rs.)
5,00,000

In [19]:
query = "Patient has critical bleeding emergency"

response = app.invoke({
    "query": query,
    "context": "",
    "answer": "",
    "intent": "",
    "escalate": False
})

print("Intent:", response["intent"])
print("\nEscalation:", response["escalate"])
print("\nAnswer:\n")
print(response["answer"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Intent: emergency

Escalation: True

Answer:

This query requires immediate human medical assistance. Your request has been escalated to a hospital support agent.
